In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.functions import broadcast
from pyspark.sql.types import *

data1 = [(1, 'Joe', 70000, 1), (2, 'Jim', 90000, 1), (3, 'Henry', 80000, 2), (4, 'Sam', 60000, 2), (5, 'Max', 90000, 1)]

schema1 = StructType([
    StructField('id',IntegerType(),False),
    StructField('name',StringType()),
    StructField('salary',IntegerType()),
    StructField('departmentId',IntegerType())
])

data2 = [(1, 'IT'), (2, 'Sales')]

schema2 = StructType([
    StructField('id',IntegerType(),False),
    StructField('name',StringType()),
])

Employee = spark.createDataFrame(data=data1,schema=schema1)
Department = spark.createDataFrame(data=data2,schema=schema2)

high = Employee.join(Department,on=Employee.departmentId==Department.id).select(Employee.name.alias('Employee'),Employee.salary.alias('salary'),Department.name.alias('Department'))

w = Window.partitionBy('Department').orderBy(f.col('salary').desc())

high = high.withColumn('rn',f.dense_rank().over(w))

high = high.filter(f.col('rn')==1).select(f.col('Department'),f.col('Employee'),f.col('salary'))

display(high)

DepartmentTop = high.write.format('delta').mode('overwrite').saveAsTable('DepartmentTop')






In [0]:
%sql
select * from DepartmentTop